### JB Hifi

In [ ]:
import time, csv, json, re, requests, os, random, shutil, glob
import pandas as pd
import numpy as np
import unicodedata
from zoneinfo import ZoneInfo
from html import unescape
from datetime import datetime, timezone
from pathlib import Path
from html import unescape
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, ElementClickInterceptedException, NoSuchElementException, TimeoutException


REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

DATA_DIR = Path(os.environ.get("JB_HIFI_DATA_DIR", REPO_ROOT.parent / "jb_hifi_private_data")).resolve()
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
BACKUP_DIR = DATA_DIR / "backups"
DAILY_DIR = RAW_DIR / "jb_daily_data"
RESULTS_SPECS = RAW_DIR / "results_specs.csv"
RESULTS_WITH_BRAND = PROCESSED_DIR / "results_with_brand.csv"
RESULTS_TAGGED = PROCESSED_DIR / "results_tagged.csv"
RESULTS_SPECS_CLEANED = PROCESSED_DIR / "results_specs_cleaned_all.csv"
MASTER_BACKUP_DIR = BACKUP_DIR / "jb_backup_data"

LISTING_CORE_COLUMNS = [
    "DateCollected", "Brand", "Title", "Price", "FullPrice", "Link", "ImageURL", "Rating", "NumRating",
]
DAILY_APPEND_KEY_COLUMNS = ["DateCollected", "Title"]
SPECS_TITLE_COLUMN = "Title"


def missing_columns(columns, required_columns):
    return [column for column in required_columns if column not in columns]


def require_columns(columns, required_columns, dataset_name):
    missing = missing_columns(columns, required_columns)
    if missing:
        raise ValueError(f"{dataset_name} is missing required columns: {missing}")


In [ ]:
# FUNCTIONS
# Setup & Utilities
def setup_driver():
    options = webdriver.ChromeOptions()
    options.add_argument("--headless")  # Run Chrome without a visible window
    return webdriver.Chrome(options=options)

# Scraping Functions
def load_all_products(driver, scroll_pause=1, click_pause=2):
    """
    Clicks the 'Load More' button repeatedly until it's gone.
    Uses JS fallback if a normal click is intercepted.
    """
    while True:
        try:
            btn = driver.find_element(By.CLASS_NAME, "load-more-button")
            
            # Scroll into view
            driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", btn)
            time.sleep(scroll_pause)

            # Try a normal click, else JS click
            try:
                btn.click()
            except ElementClickInterceptedException:
                driver.execute_script("arguments[0].click();", btn)

            time.sleep(click_pause)

        except NoSuchElementException:
            print("No more 'Load More' button")
            break

# Extract Product Data
def extract_product_data(tile):
    date_collected = datetime.now(ZoneInfo("Australia/Melbourne")).date().isoformat()

    try:
        # Core fields
        title = tile.find_element(By.CSS_SELECTOR, "[data-testid='product-card-title']").text.strip()
        link = tile.find_element(By.CSS_SELECTOR, "a.ProductCard_imageLink").get_attribute("href")
        symbol = tile.find_element(By.CSS_SELECTOR, "span[class*='PriceTag_symbol']").text.strip()
        amount = tile.find_element(By.CSS_SELECTOR, "span[class*='PriceTag_actual']").text.strip()
        image = tile.find_element(By.CSS_SELECTOR, "img").get_attribute("src")
        price = f"{symbol}{amount}".strip() if amount else "N/A"

        # Price element
        try:
            fullprice = tile.find_element(
                  By.CSS_SELECTOR,
                  "span[class*='PriceTag_symbolHeader'] + span"
               ).text.strip()
            fullprice = f"{symbol}{fullprice}"
        except NoSuchElementException:
            # if there’s no “was” price, fall back to the actual amount
            fullprice = price

        
        # Ratings
        try:
            rating_txt = tile.find_element(
                By.CSS_SELECTOR,
                "[data-testid='product-card-reviews'] "      
                "button._6zw1gn1 ._6zw1gna"                   
            ).text.strip()
            rating = float(rating_txt)
        except Exception:
            rating = None

        # Number of ratings
        try:
            num_txt = tile.find_element(By.CSS_SELECTOR,"div[class*='_6zw1gnb']").text.strip()
            # strip non-digits and convert
            num_ratings = int(re.sub(r'\D', '', num_txt))
        except (NoSuchElementException, ValueError):
            num_ratings = None
        
        # Find all tags
        tag_spans = tile.find_elements(
            By.CSS_SELECTOR,
            "span[data-testid='product-card-banner-tag'], "
            "span[data-testid^='product-card-promo-tag-']"
        )
        tags = [t.text.strip() for t in tag_spans if t.text.strip()]
        
        return {
            "date"     : date_collected,
            "title"    : title,
            "price"    : price,
            "fullprice": fullprice,
            "link"     : link,
            "image"    : image,
            "rating"   : rating,
            "num_ratings": num_ratings,
            "tags"     : tags
        }
    except NoSuchElementException:
        # If the core fields are missing, skip this tile
        return None

### FETCH SPECS INFORMATION

In [ ]:
def fetch_specs(product_url):
    headers = {"User-Agent": "Mozilla/5.0"}
    try:
        resp = requests.get(product_url, headers=headers, timeout=10)
        resp.raise_for_status()
    except Exception:
        return {}

    text = resp.text

    # Grab every window.themeConfig call
    pattern_all = r"window\.themeConfig\(\s*['\"]([^'\"]+)['\"]\s*,\s*(\{.*?\})\s*\)\s*;"
    matches = re.findall(pattern_all, text, re.DOTALL)

    # Find product.metafields
    block = None
    for key, js in matches:
        if key.strip() == "product.metafields":
            block = js
            break
    if not block:
        return {}

    # Unescape and parse JSON
    raw = unescape(block)
    try:
        data = json.loads(raw)
        specs_list = data["online_product"]["value"]["Display"]["SpecificationDetails"]
    except Exception:
        return {}

    # Flatten
    return {
        spec["Name"]: ", ".join(map(str, spec.get("Values", [])))
        for spec in specs_list
        if spec.get("Name") and spec.get("Values")
    }

### RESULTS.CSV

In [ ]:
def scrape_jbhifi_laptops(output_file="results.csv"):

    # Brand extraction function
    def extract_brand(title):
        brands = [
            "HP", "ASUS", "Dell", "Apple", "Lenovo", "MSI", "Acer", "Alienware",
            "Gigabyte", "Erazer", "Microsoft", "Samsung", "Leader", "LG", "Aftershock"
        ]

        if "lg gram" in title.lower(): return "LG"
        if "lenvo" in title.lower(): return "Lenovo"
        if "victus" in title.lower(): return "HP"
        for b in brands:
            if b.lower() in title.lower(): return b
        return "Other"

    driver = setup_driver()
    try:
        driver.get("https://www.jbhifi.com.au/collections/computers-tablets/laptops?hitsPerPage=100")
        
        load_all_products(driver)
        cards = driver.find_elements(By.CLASS_NAME, "ProductCard")

        print(f"Found {len(cards)} products.")

        rows = [d for d in (extract_product_data(c) for c in cards) if d]
    finally:
        driver.quit()

    # Determine max number of tags across all rows
    max_tags = max((len(r["tags"]) for r in rows), default=0)

    # CSV header
    core_cols = ["DateCollected", "Brand", "Title", "Price", "FullPrice", "Link", "ImageURL", "Rating", "NumRating"]
    tag_cols  = [f"Tag{i+1}" for i in range(max_tags)]
    header    = core_cols + tag_cols

    # Write CSV, padding tags lists
    with open(output_file, 'w', newline='', encoding='utf-8-sig') as f:
        writer = csv.writer(f)
        writer.writerow(header)

        for r in rows:
            brand     = extract_brand(r["title"])
            core_vals = [r["date"], brand, r["title"], r["price"], r["fullprice"], r["link"], r["image"], r["rating"], r["num_ratings"]]
            padded    = r["tags"] + [""]*(max_tags - len(r["tags"]))
            writer.writerow(core_vals + padded)

    print(f"Data saved to {output_file}!")

### SPECS.CSV

In [ ]:
def update_specs(core_file, specs_file=RESULTS_SPECS, limit=None, delay=1, jitter=0.5):
    """
    Read results.csv to get Titles & Links.
    Read existing results_specs to skip existing Titles.
    Fetch specs for new Titles, then rewrite specs_file
    with all Titles + complete set of spec columns.
    """
    # Load core data
    with open(core_file, newline='', encoding='utf-8-sig') as f:
        reader = csv.DictReader(f)
        core_rows = list(reader)
    if core_rows:
        require_columns(list(core_rows[0].keys()), [SPECS_TITLE_COLUMN, "Link"], "core listings")

    # map Title -> Link
    title_to_link = {
        unicodedata.normalize('NFKC', unescape(r["Title"].strip().lower())): r["Link"]
        for r in core_rows
    }

    # Load existing specs
    existing = {}
    if os.path.exists(specs_file):
        with open(specs_file, newline='', encoding='utf-8-sig') as f:
            rd = csv.DictReader(f)

            # Normalize headers
            rd.fieldnames = [fn.lstrip('\ufeff') for fn in rd.fieldnames]
            
            for row in rd:
                title = unicodedata.normalize('NFKC', unescape(row["Title"].strip().lower()))
                existing[title] = {
                    k: v for k, v in row.items() 
                    if k != "Title"
                }

    # Determine which Titles need specs fetched
    new_titles = [t for t in title_to_link if t not in existing]
    
    if limit:
        new_titles = new_titles[:limit]
    print(f"Found {len(new_titles)} new laptops to fetch specs for.")

    # Fetch specs for new Titles
    for i, title in enumerate(new_titles, start=1): 
        existing[title] = fetch_specs(title_to_link[title]) 
        # pause before the next request 
        wait = delay + random.random() * jitter 
        time.sleep(wait)

    # Determine full spec columns
    all_spec_keys = set()
    for specs in existing.values():
        all_spec_keys.update(specs.keys())
    all_keys = sorted(all_spec_keys)

    # Rewrite specs CSV
    with open(specs_file, 'w', newline='', encoding='utf-8-sig') as f:
        writer = csv.writer(f)
        writer.writerow(["Title"] + all_keys)

        for title, specs in existing.items():
            clean_title = unicodedata.normalize('NFKC', unescape(title))
            row_vals = []
            for key in all_keys:
                raw = specs.get(key, "")
                txt = ", ".join(map(str, raw)) if isinstance(raw, (list, tuple)) else str(raw)
                clean = unicodedata.normalize('NFKC', unescape(txt))
                row_vals.append(clean)
            
            writer.writerow([clean_title] + row_vals)

    print(f"Specs data saved to {specs_file}")

In [ ]:
today = datetime.now(ZoneInfo("Australia/Melbourne")).date().isoformat()
today

In [ ]:
if __name__ == "__main__":
    today = datetime.now(ZoneInfo("Australia/Melbourne")).date().isoformat()
    os.makedirs(DAILY_DIR, exist_ok=True)
    core_file = DAILY_DIR / f"{today}_results.csv"
    scrape_jbhifi_laptops(output_file=core_file)
    update_specs(core_file=core_file, specs_file=RESULTS_SPECS)


### COPY TO MASTER FILE results_with_brand.csv

In [ ]:
def ingest_daily(daily_file, master_file, key_cols=None, backup_dir=MASTER_BACKUP_DIR):
    key_cols = key_cols or DAILY_APPEND_KEY_COLUMNS
    df_daily  = pd.read_csv(daily_file, encoding='utf-8-sig')
    assert len(df_daily) > 0, f"{daily_file} is empty!"
    require_columns(list(df_daily.columns), LISTING_CORE_COLUMNS, "daily listings")
    require_columns(list(df_daily.columns), key_cols, "daily listings")

    df_master = pd.read_csv(master_file, encoding='utf-8-sig')
    require_columns(list(df_master.columns), LISTING_CORE_COLUMNS, "master listings")
    require_columns(list(df_master.columns), key_cols, "master listings")
    assert list(df_daily.columns) == list(df_master.columns), "Column mismatch!"

    # Dup-check on key_cols
    dupes = pd.merge(
        df_daily[key_cols].drop_duplicates(),
        df_master[key_cols].drop_duplicates(),
        on=key_cols,
        how='inner'
    )
    if not dupes.empty:
        print(f"Found {len(dupes)} duplicate rows. Aborting.")
        return

    # Backup
    today = datetime.now(ZoneInfo("Australia/Melbourne")).date().isoformat()
    bak   = f"jb_bak_{today}.csv"
    os.makedirs(backup_dir, exist_ok=True)
    bak_path = Path(backup_dir) / bak
    shutil.copy(master_file, bak_path)
    print(f"Backup saved to {bak_path}")

    old_master = len(df_master)
    print(f"Daily rows : {len(df_daily)}")
    print(f"Master rows: {old_master}")
    print(f"New rows   : {len(df_daily)} (would be appended)")

    # Append daily to master
    df_new_master = pd.concat([df_master, df_daily], ignore_index=True)

    # Write updated master
    df_new_master.to_csv(master_file, index=False, encoding='utf-8-sig')
    print(f"Appended {len(df_daily)} rows. New master row-count: {len(df_new_master)}")

    # Warning
    if len(df_new_master) != len(df_daily) + old_master:
        return "Number of rows doesn't match, DOUBLE CHECK!"
    else:
        print("Matched!")


def find_latest_daily(dir_path=DAILY_DIR, pattern="*_results.csv"):
    paths = glob.glob(str(Path(dir_path) / pattern))
    if not paths:
        raise FileNotFoundError(f"No files found in {dir_path}/{pattern}")
    return max(paths, key=os.path.getmtime)


In [ ]:
if __name__ == "__main__":
    master = RESULTS_WITH_BRAND
    daily  = find_latest_daily(DAILY_DIR)
    ingest_daily(daily, master, key_cols=DAILY_APPEND_KEY_COLUMNS, backup_dir=MASTER_BACKUP_DIR)


### PREPROCESSING

#### 1. Results.csv

In [ ]:
def tag_discontinued_products(csv_file=RESULTS_WITH_BRAND, output_file=RESULTS_TAGGED):
    """
    Loads historical listing data, adds 'Final appear day' and 'Discontinued' flags, and saves to a new file.
    """
    df = pd.read_csv(csv_file, encoding='utf-8-sig')
    require_columns(list(df.columns), LISTING_CORE_COLUMNS, "historical listings")

    # Dates
    dates = df["DateCollected"].astype(str)
    dt_iso = pd.to_datetime(dates, format="%Y-%m-%d", errors="coerce")
    dt_eu  = pd.to_datetime(dates, format="%d/%m/%Y", errors="coerce")
    # Combine, preferring ISO where available
    df["DateCollected"] = dt_iso.fillna(dt_eu)

    # Flag discontinued entries
    # Get latest scrape day in dataset
    latest_day = df["DateCollected"].max()
    # Compute last known date each product appeared
    last_seen = df.groupby("Link")["DateCollected"].max().reset_index()
    last_seen.rename(columns={"DateCollected": "Final appear day"}, inplace=True)
    df = df.merge(last_seen, on="Link", how="left")
    # Flag
    df["Discontinued"] = df["Final appear day"] < latest_day
    df["Price"] = df["Price"].str.replace(r"[\$,]", "", regex=True).astype("float64")

    # Save
    Path(output_file).parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(output_file, index=False, encoding='utf-8-sig')
    print(f"Tagged file saved to: {output_file}")


In [ ]:
tag_discontinued_products()

In [ ]:
# Double Check
csv_file = pd.read_csv(RESULTS_WITH_BRAND, encoding='utf-8-sig')
output_file = pd.read_csv(RESULTS_TAGGED, encoding='utf-8-sig')
print(csv_file.shape)
print(output_file.shape)


In [ ]:
# summary of category features
print(output_file.dtypes)
from IPython.display import display, HTML
display(HTML('<b>Table 1: Summary of numerical features</b>'))
output_file.describe().T

#### 2. Results_specs.csv

In [ ]:
import pandas as pd
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 3000)

df = pd.read_csv(RESULTS_SPECS, encoding='utf-8-sig')
require_columns(list(df.columns), [SPECS_TITLE_COLUMN], "raw specs")

print(df.shape)

empty = df.isna().any()
empty_col = empty[empty].index.tolist()
print(empty.sum())
# empty_col


In [ ]:
# Review duplicate Titles
title_lower = df["Title"].str.lower()
dup_titles = df[title_lower.duplicated(keep=False)]

if not dup_titles.empty:
    print("Duplicate Titles found:")
    print(dup_titles[["Title"]])
else:
    print("No duplicate Titles detected.")


In [ ]:
#Remove duplicates
df["Title_lower"] = df["Title"].str.lower()
df = df.drop_duplicates(subset=["Title_lower"], keep="last")
df = df.drop(columns=["Title_lower"])
print(df.shape)

In [ ]:
# Drop columns
drop_columns = ['Backlit keyboard','Contains button battery', 'PC Gaming Device', 'Microsoft Surface type', 'Tablet Type', 'Network compatibility',
                'Wi-Fi', 'Charge port', 'RAM module configuration', 'Expandable memory slot(s)', 'Expandable storage type',
                'Battery WHr', 'Battery capacity (mAh)', 'TOPS (AI metric)', 'Processor Memory Cache', 'Device screen size (inches)',
                'Response Time (ms)', 'Aspect ratio', 'Graphics memory', 'Flash storage', 'Ethernet / LAN ports', 'Intel Evo device',
                'Internal memory', "Manufacturer's Warranty", 'Memory type', 'Microsoft Surface type',	'Mini HDMI ports',	
            	'Mouse and keyboard',	'Network compatibility', 'Power supply type',	'Processor Clock Speed (GHz)',
                'Processor Max. Clock Speed (GHz)',	'Processor Memory Cache', 'RAM speeds',	'RAM type',	'Refresh Rate (Hz)',
                'Response Time (ms)',	'SSD form factor', 'Surface Connect ports', 'eMMC storage', 'Processor Model Number']
df.drop(columns = drop_columns, inplace=True)

In [ ]:
# Load old cleaned file
KEEP_EXISTING_CLEANED_VALUES = True
old = None
if KEEP_EXISTING_CLEANED_VALUES and os.path.exists(RESULTS_SPECS_CLEANED):
    old = pd.read_csv(RESULTS_SPECS_CLEANED, encoding='utf-8-sig')
    require_columns(list(old.columns), [SPECS_TITLE_COLUMN], "existing cleaned specs")

# Identify new rows
if old is not None:
    old_keys = set(old[SPECS_TITLE_COLUMN].str.lower())
    df["is_new"] = ~df[SPECS_TITLE_COLUMN].str.lower().isin(old_keys)
else:
    df["is_new"] = True


In [ ]:
# Copilot+ PC -> Boolean (blank=False)
s = df["Copilot+ PC"]
df["Copilot+ PC"] = np.where(
    s.map(lambda x: isinstance(x, bool)),      # already True/False?
    s,                                         # keep as-is
    s.astype("string").str.contains(r"\byes\b", case=False, na=False)
)

# Gaming PC flag from Title -> Boolean
df["Gaming PC"] = df["Title"].str.contains("gaming laptop", case=False, na=False)

## AI FEATURES -> standardize “Copilot keyboard key”
def clean_ai_feat(s):
    txt = str(s).lower()
    if "copilot" in txt and "keyboard" in txt:
        return "Copilot keyboard key"
    return np.nan

df["AI features"] = df["AI features"].apply(clean_ai_feat)

# PRODUCT CONDITION - Renewed
def condition(title):
    title = str(title).lower() if pd.notna(title) else ""
    if "renewed" in title:
        return "Renewed"
    if "refurbished" in title:
        return "Refurbished"
    return "Other"

df["Product condition"] = df["Title"].apply(condition)

## DISPLAY TYPE -> bucket
def bucket_display(s):
    if pd.isna(s): return np.nan
    t = s.lower()
    if "oled" in t: return "OLED"
    if "mini led" in t or "miniled" in t: return "Mini LED"
    if "ips" in t: return "IPS"
    if "tn" in t: return "TN"
    if "pixelsense" in t: return "PixelSense"
    if "led" in t or "lcd" in t: return "LCD/LED"
    if any(x in t for x in ["anti-glare","sva","uwva"]): return "Anti-glare/SVA"
    return "Other"

df["Display type"] = df["Display type"].apply(bucket_display)

# USB Ports & USB-C Ports -> numeric 
for col in ["USB Ports", "USB-C Ports"]:
    df[col] = (
        df[col]
          .fillna(0)
          .astype(int)
          .astype("Int64")
    )

## OPERATING SYSTEM
df["OS_norm"] = df["Operating system"].str.lower().str.strip()
typo_map = {
    "macos sequioa": "macos sequoia",
}

df["OS_norm"] = df["OS_norm"].replace(typo_map)

def map_os(os_str):
    if not isinstance(os_str, str):
        return "Other"
    if "chrome" in os_str: return "Chrome OS"
    if "windows 10 pro" in os_str: return "Windows 10 Pro"
    if "windows 10 home" in os_str: return "Windows 10 Home"
    if "windows 11 pro national academic" in os_str: return "Windows 11 Pro National Academic"
    if "windows 11 pro" in os_str: return "Windows 11 Pro"
    if "windows 11 home plus" in os_str: return "Windows 11 Home Plus"
    if "windows 11 home s" in os_str: return "Windows 11 Home S"
    if "windows 11 home" in os_str: return "Windows 11 Home"
    if re.match(r"windows 11(\s|$)", os_str): return "Windows 11"
    if "macos sequoia" in os_str: return "macOS Sequoia"
    if "macos ventura" in os_str: return "macOS Ventura"
    if "macos monterey" in os_str: return "macOS Monterey"
    if "macos sonoma" in os_str: return "macOS Sonoma"
    return "Other"

df["Operating system"] = df["OS_norm"].apply(map_os)

def map_family(os_clean):
    if os_clean.startswith("Windows"): return "Windows"
    if os_clean.startswith("macOS"): return "macOS"
    if os_clean == "Chrome OS": return "Chrome OS"
    return "Other"

df["OS_family"] = df["Operating system"].apply(map_family)


## MONITOR RESOLUTION
def clean_resolution(raw):
    """
    Normalize a resolution string into "WIDTH x HEIGHT".  
    Returns NaN if it can’t parse two integers.
    """
    if pd.isna(raw):
        return pd.NA
    s = str(raw).strip().lower()

    # unify delimiters
    s = re.sub(r'(–|×|by)', 'x', s)
    m = re.search(r'(\d{3,4})\s*x\s*(\d{3,4})', s)
    if not m:
        return pd.NA

    w, h = m.group(1), m.group(2)
    return f"{w} x {h}"

# Merge resolutions -> Monitor resolution priority
merged_res = df["Monitor resolution"].fillna(df["Resolution (Pixels)"])
df["Monitor resolution"] = merged_res.apply(clean_resolution)
df = df.drop(columns=["Resolution (Pixels)"])
# # Split into two numeric columns:
# df[["Res_width", "Res_height"]] = (
#     df["Monitor_res_clean"]
#       .str.split(" x ", expand=True)
#       .astype("Int64")
# )

## PROCESSOR TYPE -> group and bucket by brand
def clean_processor_type(raw):
    if pd.isna(raw):
        return pd.NA
    s = str(raw)
    s = re.sub(r'®|™|\(.*?\)|\[.*?\]', '', s) # Remove trademarks and stray punctuation
    s = re.sub(r'coretm', 'core', s, flags=re.IGNORECASE) 
    s = re.sub(r'\s+', ' ', s).strip()
    return s 

def bucket_processor_brand(ptype):
    if isinstance(ptype, str):
        t = ptype.lower()
        t = re.sub(r'®|™|\(.*?\)|\[.*?\]', '', t)
        if 'intel' in t: return 'Intel'
        if 'qualcomm' in t: return 'Qualcomm'
        if t.startswith('apple m'): return 'Apple M'
        if 'ryzen' in t or 'athlon' in t: return 'AMD'
        if t.startswith('mtk'): return 'MTK'
    return 'Other'

df['Processor Type'] = df['Processor Type'].apply(clean_processor_type)
df['Processor brand'] = df['Processor Type'].apply(bucket_processor_brand)

## GRAPHICS PROCESSOR -> buckets

def get_gpu_bucket(row):

    # Check NA
    if pd.isna(row["Graphics processor"]) and not pd.isna(row["Graphics card series"]): return row["Graphics card series"]

    title = str(row["Title"]).lower()
    t = str(row["Graphics processor"]).lower()
    gpu_card = str(row["Graphics card series"]).lower()
    
    match = re.search(r"rtx[\s\-]?(\d{3,4}[ti]?)", t)
    if match:
        return f"NVIDIA RTX {match.group(1)}"
    if "iris" in t and "xe" in t: return "Intel Iris Xe"
    if "uhd" in t and "intel" in t: return "Intel UHD"
    if "hd graphics" in t or "intel hd" in t: return "Intel HD"
    if "pentium gold" in t: return "Intel Pentium Gold"
    if "arc" in t: return "Intel Arc"
    if "radeon" in t: return "AMD Radeon"
    if "adreno" in t: return "Qualcomm Adreno"
    if "mali" in t: return "ARM Mali"
    if "halo" in t: return "AMD Strix Halo"
    elif "intel" in t and "graphic" in t: return "Intel Graphics"
    elif "integrated" in t: return "Integrated"
    if re.search(r"\d+\s*core gpu", t): return "Apple GPU"
    elif "apple" in title: return "Apple GPU"

    return "Other"

df["Graphics processor"] = df.apply(get_gpu_bucket, axis=1)

## GPU BRAND
def bucket_gpu_brand(raw):
    if pd.isna(raw):
        return pd.NA
    t = str(raw).lower()
    t = re.sub(r'®|™|\(.*?\)|\[.*?\]', '', t)
    if "intel" in t: return "Intel"
    if any(x in t for x in ("nvidia", "rtx", "gtx", "geforce")): return "NVIDIA"
    if any(x in t for x in ("amd", "radeon")): return "AMD"
    if "qualcomm" in t or "adreno" in t: return "Qualcomm"
    if "apple" in t: return "Apple"
    if "arm" in t or "mali" in t: return "ARM"
    if "integrated" in t: return "Integrated"

    return "Other"

df["GPU brand"] = df["Graphics processor"].apply(bucket_gpu_brand)


# Restore old cleaned values for existing rows
if KEEP_EXISTING_CLEANED_VALUES and old is not None:
    merged = df.merge(old, on=SPECS_TITLE_COLUMN, how="left", suffixes=("", "_old"))
    for col in df.columns:
        old_col = col + "_old"
        if old_col in merged.columns:
            merged[col] = merged[col].where(merged["is_new"], merged[old_col])
    df = merged[df.columns]

# Save final cleaned dataset
Path(RESULTS_SPECS_CLEANED).parent.mkdir(parents=True, exist_ok=True)
df.to_csv(RESULTS_SPECS_CLEANED, index=False, encoding="utf-8-sig")
print(f"All transformations applied and saved to {RESULTS_SPECS_CLEANED}")

In [ ]:
# Double check
results_specs = pd.read_csv(RESULTS_SPECS, encoding='utf-8-sig')
results_specs_cleaned_all = pd.read_csv(RESULTS_SPECS_CLEANED, encoding='utf-8-sig')
print(results_specs.shape)
print(results_specs_cleaned_all.shape)


In [ ]:
# Check for constant features
df = results_specs_cleaned_all
constant = df.nunique() == 1
print(f'The number of constant features within the dataset is {constant.sum()}.')
constant_col = constant[constant].index.tolist()
for c in constant_col:
    print(f"Constant col: {c}")
    print(df[c].unique())
    print('')

In [ ]:
print(df.dtypes.to_string())

In [ ]:
obj_cols = [
    col for col in df.select_dtypes(include="object").columns.tolist()
    if col != "Title"
]
for c in obj_cols:
    print(f"Unique value for '{c}'")
    print(df[c].unique())
    print('')